In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error


# ============================================================
# CONFIGURATION
# ============================================================

LOOK_BACK = 250
HORIZON = 20
SLIDE = 1

QUANTILE_LEVELS = np.arange(0.1, 1.0, 0.1)

VALIDATION_FRACTION = 0.15
PURGE_VALIDATION = True
RANDOM_SEED = 42

# Set paths
DATA_DIR = Path(f"BForTFin_log_return/arima_split_datasets/output_{REVIEW_FREQUENCY}")
RESULTS_DIR = Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}/lightgbm_standardized_train_test")

ASSET_FOLDERS = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

MODEL_NAME = "LightGBM"
MODEL_SUFFIX = "_lightgbm"

# False = reuse existing saved point/quantile forecasts if present.
# This makes the script safe to resume after interruption.
OVERWRITE = False

# Small shared structural grid.
# One configuration is selected per dataset using chronological validation
# on the point-forecasting task, then reused for point + quantile models.
CANDIDATE_PARAMS = [
    {
        "num_leaves": 15,
        "learning_rate": 0.05,
        "n_estimators": 100,
    },
    {
        "num_leaves": 31,
        "learning_rate": 0.05,
        "n_estimators": 100,
    },
    {
        "num_leaves": 31,
        "learning_rate": 0.05,
        "n_estimators": 200,
    },
    {
        "num_leaves": 63,
        "learning_rate": 0.05,
        "n_estimators": 100,
    },
]


# ============================================================
# METRICS
# ============================================================

def quantile_crps(
    y,
    quantile_forecasts,
    quantile_levels=QUANTILE_LEVELS,
):
    """
    y:
        shape (...,)

    quantile_forecasts:
        shape (..., Q)

    returns:
        CRPS approximation with shape (...)
    """
    y = np.asarray(y, dtype=float)
    qf = np.asarray(quantile_forecasts, dtype=float)
    levels = np.asarray(quantile_levels, dtype=float)

    error = y[..., None] - qf

    pinball = np.maximum(
        levels * error,
        (levels - 1.0) * error,
    )

    return 2.0 * np.mean(pinball, axis=-1)


def calculate_mase_from_point_forecast(X, Y_true, Y_pred):
    """
    X:      (N, 250) original-scale context
    Y_true: (N, 20)  original-scale targets
    Y_pred: (N, 20)  original-scale point forecasts
    """
    if X.shape[0] != Y_true.shape[0]:
        raise ValueError(
            f"X and Y_true have different N: "
            f"{X.shape[0]} vs {Y_true.shape[0]}"
        )

    if Y_pred.shape != Y_true.shape:
        raise ValueError(
            f"Prediction shape {Y_pred.shape} "
            f"does not match true Y shape {Y_true.shape}"
        )

    naive_forecast = X[:, -1][:, None]

    model_mae_by_horizon = np.mean(
        np.abs(Y_true - Y_pred),
        axis=0,
    )

    naive_mae_by_horizon = np.mean(
        np.abs(Y_true - naive_forecast),
        axis=0,
    )

    mase_by_horizon = np.full(
        Y_true.shape[1],
        np.nan,
        dtype=float,
    )

    valid_horizons = naive_mae_by_horizon > 0

    mase_by_horizon[valid_horizons] = (
        model_mae_by_horizon[valid_horizons]
        / naive_mae_by_horizon[valid_horizons]
    )

    mean_mase = np.nanmean(mase_by_horizon)

    return (
        mean_mase,
        mase_by_horizon,
        model_mae_by_horizon,
        naive_mae_by_horizon,
    )


# ============================================================
# DATA / SCALING HELPERS
# ============================================================

def reconstruct_chronological_series(
    x: np.ndarray,
    y: np.ndarray,
    slide: int,
) -> np.ndarray:
    """
    Reconstruct one univariate chronological series from overlapping
    supervised-learning rows.

    For slide=1, the first X window is followed by the final newly observed
    value from each subsequent X row, then the final Y row.
    """
    if x.ndim != 2 or y.ndim != 2:
        raise ValueError("x and y must both be 2-D arrays.")

    if len(x) == 0:
        raise ValueError("Cannot reconstruct a series from empty arrays.")

    reconstructed = list(x[0])

    for row_index in range(1, len(x)):
        reconstructed.extend(x[row_index, -slide:])

    reconstructed.extend(y[-1])

    return np.asarray(reconstructed, dtype=np.float64)


def fit_univariate_scaler(
    X: np.ndarray,
    Y: np.ndarray,
    slide: int = SLIDE,
) -> StandardScaler:
    """
    Fit ONE StandardScaler to the chronological training series only.

    This gives a single mean/std for the univariate time series so X, Y,
    point forecasts, and quantiles can all be transformed consistently.
    """
    training_series = reconstruct_chronological_series(
        X,
        Y,
        slide=slide,
    )

    scaler = StandardScaler()
    scaler.fit(training_series.reshape(-1, 1))

    return scaler


def transform_matrix(
    scaler: StandardScaler,
    values: np.ndarray,
) -> np.ndarray:
    values = np.asarray(values, dtype=float)

    return scaler.transform(
        values.reshape(-1, 1)
    ).reshape(values.shape)


def inverse_transform_matrix(
    scaler: StandardScaler,
    values: np.ndarray,
) -> np.ndarray:
    values = np.asarray(values, dtype=float)

    return scaler.inverse_transform(
        values.reshape(-1, 1)
    ).reshape(values.shape)


def inverse_transform_quantiles(
    scaler: StandardScaler,
    quantiles: np.ndarray,
) -> np.ndarray:
    """
    quantiles shape: (N, H, Q)

    StandardScaler is a monotone affine transform, so every quantile can be
    inverse-transformed with the same fitted scaler.
    """
    quantiles = np.asarray(quantiles, dtype=float)

    return scaler.inverse_transform(
        quantiles.reshape(-1, 1)
    ).reshape(quantiles.shape)


def load_supervised_pair(
    train_path: Path,
    test_path: Path,
):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    x_cols = [f"X_{i}" for i in range(LOOK_BACK)]
    y_cols = [f"Y_{i}" for i in range(HORIZON)]

    required = x_cols + y_cols

    for name, df in (
        ("train", train_df),
        ("test", test_df),
    ):
        missing = [col for col in required if col not in df.columns]

        if missing:
            raise ValueError(
                f"{name} file {train_path.name if name == 'train' else test_path.name} "
                f"is missing columns: {missing}"
            )

    trainX = train_df[x_cols].to_numpy(dtype=float)
    trainY = train_df[y_cols].to_numpy(dtype=float)
    testX = test_df[x_cols].to_numpy(dtype=float)
    testY = test_df[y_cols].to_numpy(dtype=float)

    for array_name, array in (
        ("trainX", trainX),
        ("trainY", trainY),
        ("testX", testX),
        ("testY", testY),
    ):
        if not np.isfinite(array).all():
            raise ValueError(
                f"{array_name} contains NaN or infinite values."
            )

    return (
        train_df,
        test_df,
        trainX,
        trainY,
        testX,
        testY,
        x_cols,
        y_cols,
    )


# ============================================================
# CHRONOLOGICAL VALIDATION
# ============================================================

def make_validation_split(
    trainX: np.ndarray,
    trainY: np.ndarray,
):
    number_of_train_rows = len(trainX)

    number_of_validation_rows = max(
        1,
        int(
            math.ceil(
                number_of_train_rows * VALIDATION_FRACTION
            )
        ),
    )

    validation_start = (
        number_of_train_rows
        - number_of_validation_rows
    )

    if PURGE_VALIDATION:
        purge_rows = max(
            0,
            math.ceil(HORIZON / SLIDE) - 1,
        )
    else:
        purge_rows = 0

    model_fit_end = validation_start - purge_rows

    if model_fit_end < 2:
        raise ValueError(
            "Insufficient training rows after chronological validation "
            f"split: total={number_of_train_rows}, "
            f"validation={number_of_validation_rows}, "
            f"purged={purge_rows}"
        )

    model_fit_x = trainX[:model_fit_end]
    model_fit_y = trainY[:model_fit_end]

    validation_x = trainX[validation_start:]
    validation_y = trainY[validation_start:]

    return {
        "model_fit_x": model_fit_x,
        "model_fit_y": model_fit_y,
        "validation_x": validation_x,
        "validation_y": validation_y,
        "model_fit_rows": int(model_fit_end),
        "validation_rows": int(number_of_validation_rows),
        "purged_rows": int(purge_rows),
        "validation_start": int(validation_start),
    }


def base_lightgbm_kwargs():
    return {
        "random_state": RANDOM_SEED,
        "n_jobs": -1,
        "verbosity": -1,
    }


def select_lightgbm_params(
    trainX: np.ndarray,
    trainY: np.ndarray,
):
    """
    Select ONE structural LightGBM configuration per dataset.

    Selection criterion:
        average validation MSE across all 20 horizons,
        evaluated on ORIGINAL scale.

    Scaling during model selection is fitted only on the model-fitting
    subset, never on validation or test data.
    """
    split = make_validation_split(
        trainX,
        trainY,
    )

    model_fit_x = split["model_fit_x"]
    model_fit_y = split["model_fit_y"]
    validation_x = split["validation_x"]
    validation_y = split["validation_y"]

    selection_scaler = fit_univariate_scaler(
        model_fit_x,
        model_fit_y,
        slide=SLIDE,
    )

    model_fit_x_scaled = transform_matrix(
        selection_scaler,
        model_fit_x,
    )
    model_fit_y_scaled = transform_matrix(
        selection_scaler,
        model_fit_y,
    )
    validation_x_scaled = transform_matrix(
        selection_scaler,
        validation_x,
    )

    candidate_results = []

    for candidate_index, params in enumerate(
        CANDIDATE_PARAMS,
        start=1,
    ):
        print(
            f"    Candidate {candidate_index}/{len(CANDIDATE_PARAMS)}: "
            f"{params}"
        )

        validation_pred_scaled = np.zeros_like(
            validation_y,
            dtype=float,
        )

        for h in range(HORIZON):
            model = LGBMRegressor(
                objective="regression",
                **params,
                **base_lightgbm_kwargs(),
            )

            model.fit(
                model_fit_x_scaled,
                model_fit_y_scaled[:, h],
            )

            validation_pred_scaled[:, h] = model.predict(
                validation_x_scaled
            )

        validation_pred = inverse_transform_matrix(
            selection_scaler,
            validation_pred_scaled,
        )

        validation_mse_by_horizon = np.mean(
            (validation_y - validation_pred) ** 2,
            axis=0,
        )

        mean_validation_mse = float(
            np.mean(validation_mse_by_horizon)
        )

        candidate_results.append({
            "params": dict(params),
            "mean_validation_mse": mean_validation_mse,
            "validation_mse_by_horizon": (
                validation_mse_by_horizon.tolist()
            ),
        })

        print(
            f"      mean validation MSE = "
            f"{mean_validation_mse:.10f}"
        )

    best_result = min(
        candidate_results,
        key=lambda item: item["mean_validation_mse"],
    )

    return (
        best_result["params"],
        candidate_results,
        split,
    )


# ============================================================
# FINAL LIGHTGBM FITS
# ============================================================

def fit_point_models(
    trainX_scaled: np.ndarray,
    trainY_scaled: np.ndarray,
    testX_scaled: np.ndarray,
    selected_params: dict,
) -> np.ndarray:
    """
    Fit one direct LightGBM point model per horizon.
    Returns standardized predictions of shape (N_test, 20).
    """
    predictions_scaled = np.zeros(
        (len(testX_scaled), HORIZON),
        dtype=float,
    )

    for h in range(HORIZON):
        print(
            f"    Point model horizon "
            f"{h + 1}/{HORIZON}"
        )

        model = LGBMRegressor(
            objective="regression",
            **selected_params,
            **base_lightgbm_kwargs(),
        )

        model.fit(
            trainX_scaled,
            trainY_scaled[:, h],
        )

        predictions_scaled[:, h] = model.predict(
            testX_scaled
        )

    return predictions_scaled


def fit_quantile_models(
    trainX_scaled: np.ndarray,
    trainY_scaled: np.ndarray,
    testX_scaled: np.ndarray,
    selected_params: dict,
) -> np.ndarray:
    """
    Fit one direct LightGBM quantile model for every horizon x quantile.

    Returns standardized quantile forecasts:
        shape (N_test, 20, 9)
    """
    q_predictions_scaled = np.zeros(
        (
            len(testX_scaled),
            HORIZON,
            len(QUANTILE_LEVELS),
        ),
        dtype=float,
    )

    for h in range(HORIZON):
        print(
            f"    Quantile horizon "
            f"{h + 1}/{HORIZON}"
        )

        for q_idx, q in enumerate(QUANTILE_LEVELS):
            model = LGBMRegressor(
                objective="quantile",
                alpha=float(q),
                **selected_params,
                **base_lightgbm_kwargs(),
            )

            model.fit(
                trainX_scaled,
                trainY_scaled[:, h],
            )

            q_predictions_scaled[:, h, q_idx] = (
                model.predict(testX_scaled)
            )

    return q_predictions_scaled

def enforce_non_crossing(quantile_predictions):
    """
    Enforce monotonically increasing quantile forecasts.

    Input shape:
        (N, H, Q)

    For every forecast origin and horizon:
        Q_0.1 <= Q_0.2 <= ... <= Q_0.9
    """
    quantile_predictions = np.asarray(
        quantile_predictions,
        dtype=float,
    )

    return np.sort(
        quantile_predictions,
        axis=-1,
    )

def quantify_quantile_crossing(quantiles):
    """
    quantiles shape: (N, H, Q)
    """

    differences = np.diff(
        quantiles,
        axis=-1,
    )

    crossing_mask = differences < 0

    n_crossing_pairs = np.sum(crossing_mask)

    n_total_pairs = crossing_mask.size

    crossing_rate = (
        n_crossing_pairs / n_total_pairs
    )

    # Forecast origin/horizon combinations having
    # at least one crossing
    any_crossing = np.any(
        crossing_mask,
        axis=-1,
    )

    affected_forecasts = np.sum(any_crossing)

    total_forecasts = any_crossing.size

    affected_rate = (
        affected_forecasts / total_forecasts
    )

    return {
        "crossing_pairs": int(n_crossing_pairs),
        "total_pairs": int(n_total_pairs),
        "crossing_pair_rate": float(crossing_rate),
        "affected_forecasts": int(affected_forecasts),
        "total_forecasts": int(total_forecasts),
        "affected_forecast_rate": float(affected_rate),
    }

# ============================================================
# OUTPUT HELPERS
# ============================================================

def dataset_display_name(test_stem: str) -> str:
    """
    Example:
      test_Bitcoin Historical Data_no_preprocess_lookback-250_horizon-20_slide-1
    ->
      Bitcoin Historical Data
    """
    prefix = "test_"
    suffix = (
        f"_no_preprocess"
        f"_lookback-{LOOK_BACK}"
        f"_horizon-{HORIZON}"
        f"_slide-{SLIDE}"
    )

    name = test_stem

    if name.startswith(prefix):
        name = name[len(prefix):]

    if name.endswith(suffix):
        name = name[:-len(suffix)]

    return name


def make_output_paths(
    model_dir: Path,
    test_stem: str,
):
    base = model_dir / f"{test_stem}{MODEL_SUFFIX}"

    return {
        "point": Path(f"{base}.csv"),
        "quantiles": Path(f"{base}_quantiles.npz"),
        "crps": Path(f"{base}_crps.npz"),
        "crps_by_horizon": Path(
            f"{base}_crps_by_horizon.csv"
        ),
        "mase_by_horizon": Path(
            f"{base}_mase_by_horizon.csv"
        ),
        "metadata": Path(
            f"{base}_training.json"
        ),
    }


def save_evaluation_outputs(
    paths: dict,
    test_df: pd.DataFrame,
    x_cols: list,
    y_cols: list,
    testX_original: np.ndarray,
    testY_original: np.ndarray,
    point_predictions_original: np.ndarray,
    quantile_predictions_original: np.ndarray,
):
    # --------------------------------------------------------
    # Point forecast CSV
    # --------------------------------------------------------
    point_df = test_df[x_cols].copy()

    for h, col in enumerate(y_cols):
        point_df[col] = point_predictions_original[:, h]

    point_df.to_csv(
        paths["point"],
        index=False,
    )

    # --------------------------------------------------------
    # Quantile forecast NPZ
    # --------------------------------------------------------
    np.savez_compressed(
        paths["quantiles"],
        quantiles=quantile_predictions_original,
        quantile_levels=QUANTILE_LEVELS,
    )

    # --------------------------------------------------------
    # qCRPS on ORIGINAL scale
    # --------------------------------------------------------
    crps_matrix = quantile_crps(
        testY_original,
        quantile_predictions_original,
        QUANTILE_LEVELS,
    )


    mean_qcrps_by_horizon = np.mean(
        crps_matrix,
        axis=0,
    )

    mean_qcrps = float(
        np.mean(mean_qcrps_by_horizon)
    )

    # Match existing format: one key named "crps".
    np.savez_compressed(
        paths["crps"],
        crps=crps_matrix,
    )

    crps_horizon_df = pd.DataFrame({
        "horizon": np.arange(1, HORIZON + 1),
        "mean_qcrps": mean_qcrps_by_horizon,
    })

    crps_horizon_df.to_csv(
        paths["crps_by_horizon"],
        index=False,
    )

    # --------------------------------------------------------
    # MASE on ORIGINAL scale
    # --------------------------------------------------------
    (
        mean_mase,
        mase_by_horizon,
        model_mae_by_horizon,
        naive_mae_by_horizon,
    ) = calculate_mase_from_point_forecast(
        testX_original,
        testY_original,
        point_predictions_original,
    )

    mase_df = pd.DataFrame({
        "horizon": np.arange(1, HORIZON + 1),
        "model_mae": model_mae_by_horizon,
        "naive_mae": naive_mae_by_horizon,
        "mase": mase_by_horizon,
    })

    mase_df.to_csv(
        paths["mase_by_horizon"],
        index=False,
    )

    return {
        "mean_qcrps": mean_qcrps,
        "mean_mase": float(mean_mase),
        "crps_matrix": crps_matrix,
    }


# ============================================================
# ONE DATASET
# ============================================================

def process_dataset(
    train_path: Path,
    test_path: Path,
    model_dir: Path,
    asset_class: str,
):
    (
        train_df,
        test_df,
        trainX,
        trainY,
        testX,
        testY,
        x_cols,
        y_cols,
    ) = load_supervised_pair(
        train_path,
        test_path,
    )

    test_stem = test_path.stem
    dataset_name = dataset_display_name(
        test_stem
    )

    paths = make_output_paths(
        model_dir,
        test_stem,
    )

    print("\n" + "=" * 80)
    print(f"DATASET: {dataset_name}")
    print(f"ASSET CLASS: {asset_class}")
    print(f"TRAIN ROWS: {len(train_df)}")
    print(f"TEST ROWS:  {len(test_df)}")
    print("=" * 80)

    # --------------------------------------------------------
    # Reuse completed forecasts if possible
    # --------------------------------------------------------
    can_reuse = (
        not OVERWRITE
        and paths["point"].exists()
        and paths["quantiles"].exists()
    )

    if can_reuse:
        print("  Reusing existing point + quantile forecasts.")

        point_df = pd.read_csv(
            paths["point"]
        )

        point_predictions = point_df[
            y_cols
        ].to_numpy(dtype=float)

        quantile_file = np.load(
            paths["quantiles"]
        )

        quantile_predictions = np.asarray(
            quantile_file["quantiles"],
            dtype=float,
        )

        selected_params = None
        candidate_results = None
        split_info = None
        final_scaler = None

        
    else:
        # ----------------------------------------------------
        # Hyperparameter selection inside training partition
        # ----------------------------------------------------
        print("  Selecting LightGBM hyperparameters...")

        (
            selected_params,
            candidate_results,
            split_info,
        ) = select_lightgbm_params(
            trainX,
            trainY,
        )

        print(
            f"  Selected params: {selected_params}"
        )

        # ----------------------------------------------------
        # FINAL scaler: fit on entire 80% training partition
        # ----------------------------------------------------
        final_scaler = fit_univariate_scaler(
            trainX,
            trainY,
            slide=SLIDE,
        )

        trainX_scaled = transform_matrix(
            final_scaler,
            trainX,
        )
        trainY_scaled = transform_matrix(
            final_scaler,
            trainY,
        )
        testX_scaled = transform_matrix(
            final_scaler,
            testX,
        )

        # ----------------------------------------------------
        # Point models
        # ----------------------------------------------------
        print("  Fitting final point models...")

        point_predictions_scaled = fit_point_models(
            trainX_scaled,
            trainY_scaled,
            testX_scaled,
            selected_params,
        )

        # INVERSE TRANSFORM BEFORE MASE / SAVING
        point_predictions = inverse_transform_matrix(
            final_scaler,
            point_predictions_scaled,
        )

        # ----------------------------------------------------
        # Quantile models
        # ----------------------------------------------------
        print("  Fitting final quantile models...")

        quantile_predictions_scaled = fit_quantile_models(
            trainX_scaled,
            trainY_scaled,
            testX_scaled,
            selected_params,
        )

        # INVERSE TRANSFORM BEFORE qCRPS / SAVING
        quantile_predictions = inverse_transform_quantiles(
            final_scaler,
            quantile_predictions_scaled,
        )

    # Enforce non-crossing quantiles by monotone rearrangement
    #quantile_predictions = enforce_non_crossing(
       # quantile_predictions
    #)

    # ============================================================
    # QUANTILE CROSSING CORRECTION
    # ============================================================
    
    crossing_mask = np.diff(
        quantile_predictions,
        axis=-1,
    ) < 0
    
    crossing_pair_rate = np.mean(crossing_mask)
    
    affected_forecast_rate = np.mean(
        np.any(crossing_mask, axis=-1)
    )
    
    print(
        f"  Crossing pair rate before correction: "
        f"{crossing_pair_rate:.2%}"
    )
    
    print(
        f"  Forecasts with >=1 crossing: "
        f"{affected_forecast_rate:.2%}"
    )
    
    # Monotone rearrangement
    quantile_predictions = np.sort(
        quantile_predictions,
        axis=-1,
    )
    
    assert np.all(
        np.diff(
            quantile_predictions,
            axis=-1,
        ) >= 0
    )
    
    print("  Quantile crossing corrected.")
    
    # --------------------------------------------------------
    # Sanity checks
    # --------------------------------------------------------
    expected_point_shape = (
        len(test_df),
        HORIZON,
    )

    expected_quantile_shape = (
        len(test_df),
        HORIZON,
        len(QUANTILE_LEVELS),
    )

    if point_predictions.shape != expected_point_shape:
        raise ValueError(
            f"Point prediction shape "
            f"{point_predictions.shape} != "
            f"{expected_point_shape}"
        )

    if quantile_predictions.shape != expected_quantile_shape:
        raise ValueError(
            f"Quantile prediction shape "
            f"{quantile_predictions.shape} != "
            f"{expected_quantile_shape}"
        )

    if not np.isfinite(point_predictions).all():
        raise ValueError(
            "Point forecasts contain non-finite values."
        )

    if not np.isfinite(quantile_predictions).all():
        raise ValueError(
            "Quantile forecasts contain non-finite values."
        )

    # --------------------------------------------------------
    # Save forecasts + evaluate on ORIGINAL scale
    # --------------------------------------------------------
    metrics = save_evaluation_outputs(
        paths=paths,
        test_df=test_df,
        x_cols=x_cols,
        y_cols=y_cols,
        testX_original=testX,
        testY_original=testY,
        point_predictions_original=point_predictions,
        quantile_predictions_original=quantile_predictions,
    )

    print(
        f"  Mean qCRPS: "
        f"{metrics['mean_qcrps']:.10f}"
    )

    print(
        f"  Mean MASE:  "
        f"{metrics['mean_mase']:.10f}"
    )

    # --------------------------------------------------------
    # Save training metadata
    # --------------------------------------------------------
    if not can_reuse:
        metadata = {
            "model": MODEL_NAME,
            "look_back": LOOK_BACK,
            "horizon": HORIZON,
            "slide": SLIDE,
            "random_seed": RANDOM_SEED,
            "outer_split": "precomputed chronological 80/20 train-test split",
            "validation_fraction_of_training": VALIDATION_FRACTION,
            "purge_validation": PURGE_VALIDATION,
            "model_fit_rows": split_info["model_fit_rows"],
            "validation_rows": split_info["validation_rows"],
            "purged_rows": split_info["purged_rows"],
            "scaling": {
                "type": "StandardScaler",
                "fit_scope": (
                    "single univariate scaler fitted only on the "
                    "chronological training series"
                ),
                "final_mean": float(
                    final_scaler.mean_[0]
                ),
                "final_scale": float(
                    final_scaler.scale_[0]
                ),
                "inverse_transform_before_evaluation": True,
            },
            "selected_params": selected_params,
            "candidate_results": candidate_results,
            "point_forecasting": {
                "strategy": (
                    "20 independent direct regression models, "
                    "one per horizon"
                ),
                "objective": "regression",
            },
            "probabilistic_forecasting": {
                "strategy": (
                    "independent quantile regression per horizon "
                    "and quantile level"
                ),
                "objective": "quantile",
                "quantile_levels": (
                    QUANTILE_LEVELS.tolist()
                ),
                "n_quantile_models": (
                    HORIZON
                    * len(QUANTILE_LEVELS)
                ),
                "sampling": (
                    "not used; LightGBM directly predicts "
                    "quantiles"
                ),
            },
        }

        with open(
            paths["metadata"],
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                metadata,
                file,
                indent=2,
            )

    # --------------------------------------------------------
    # Summary row
    # --------------------------------------------------------
    summary_row = {
        "asset_class": asset_class,
        "dataset": dataset_name,
        "model": MODEL_NAME,
        "mean_qcrps": metrics["mean_qcrps"],
        "mean_mase": metrics["mean_mase"],
        "n_total_windows": (
            len(train_df) + len(test_df)
        ),
        "n_test_windows": len(test_df),
    }

    return summary_row


# ============================================================
# ALL ASSET CLASSES / DATASETS
# ============================================================

def run_all():
    RESULTS_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    summary_path = (
        RESULTS_DIR
        / "lightgbm_train_test_summary_all_datasets.csv"
    )

    summary_rows = []

    # If resuming, keep already completed summary rows.
    if summary_path.exists() and not OVERWRITE:
        old_summary = pd.read_csv(
            summary_path
        )

        if len(old_summary) > 0:
            summary_rows = old_summary.to_dict(
                orient="records"
            )

    completed_keys = {
        (
            row["asset_class"],
            row["dataset"],
        )
        for row in summary_rows
    }

    for folder_name, asset_class in ASSET_FOLDERS.items():
        input_dir = DATA_DIR / folder_name
        model_dir = RESULTS_DIR / folder_name

        model_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        if not input_dir.exists():
            print(
                f"WARNING: Missing input folder: "
                f"{input_dir}"
            )
            continue

        pattern = (
            f"train_*_no_preprocess"
            f"_lookback-{LOOK_BACK}"
            f"_horizon-{HORIZON}"
            f"_slide-{SLIDE}.csv"
        )

        train_paths = sorted(
            input_dir.glob(pattern)
        )

        print("\n" + "#" * 80)
        print(
            f"ASSET CLASS: {asset_class} "
            f"({folder_name})"
        )
        print(
            f"FOUND {len(train_paths)} TRAIN FILES"
        )
        print("#" * 80)

        for train_path in train_paths:
            test_name = (
                "test_"
                + train_path.name[len("train_"):]
            )

            test_path = input_dir / test_name

            if not test_path.exists():
                print(
                    f"WARNING: Missing matching test file "
                    f"for {train_path.name}"
                )
                continue

            dataset_name = dataset_display_name(
                test_path.stem
            )

            key = (
                asset_class,
                dataset_name,
            )

            # If summary + output files already exist, skip cleanly.
            paths = make_output_paths(
                model_dir,
                test_path.stem,
            )

            if (
                not OVERWRITE
                and key in completed_keys
                and paths["point"].exists()
                and paths["quantiles"].exists()
                and paths["crps"].exists()
                and paths["crps_by_horizon"].exists()
                and paths["mase_by_horizon"].exists()
            ):
                print(
                    f"\nSKIP completed dataset: "
                    f"{dataset_name}"
                )
                continue

            try:
                summary_row = process_dataset(
                    train_path=train_path,
                    test_path=test_path,
                    model_dir=model_dir,
                    asset_class=asset_class,
                )

                # Replace any older row for this dataset.
                summary_rows = [
                    row
                    for row in summary_rows
                    if not (
                        row["asset_class"]
                        == summary_row["asset_class"]
                        and row["dataset"]
                        == summary_row["dataset"]
                    )
                ]

                summary_rows.append(
                    summary_row
                )

                # Checkpoint after every dataset.
                summary_df = pd.DataFrame(
                    summary_rows
                )

                summary_df = summary_df[
                    [
                        "asset_class",
                        "dataset",
                        "model",
                        "mean_qcrps",
                        "mean_mase",
                        "n_total_windows",
                        "n_test_windows",
                    ]
                ]

                summary_df.to_csv(
                    summary_path,
                    index=False,
                )

                completed_keys.add(
                    (
                        summary_row["asset_class"],
                        summary_row["dataset"],
                    )
                )

                print(
                    f"  Updated summary: "
                    f"{summary_path}"
                )

            except Exception as exc:
                print(
                    f"\nERROR processing "
                    f"{train_path.name}:"
                )
                print(repr(exc))

                # Continue to next dataset instead of losing the batch.
                continue

    final_summary = pd.DataFrame(
        summary_rows
    )

    if len(final_summary) > 0:
        final_summary = final_summary[
            [
                "asset_class",
                "dataset",
                "model",
                "mean_qcrps",
                "mean_mase",
                "n_total_windows",
                "n_test_windows",
            ]
        ]

        final_summary.to_csv(
            summary_path,
            index=False,
        )

    print("\nDONE.")
    print(
        f"Summary saved to: "
        f"{summary_path}"
    )

    return final_summary


if __name__ == "__main__":
    summary = run_all()
    print(summary)
